In [1]:
# =========================================================
# PS6E4 - Sudhanshu-style XGB Multi-Seed Reproduction
#
# Original design:
# - External orig data
# - Domain/formula features
# - ngram categorical interactions
# - binned numeric x categorical interactions
# - numeric-category aggregation features
# - rounding / digit / decimal / bins
# - pairwise interactions
# - TargetEncoder multiclass
# - XGBoost categorical
# - log-odds bias tuning
#
# Multi seeds:
#   SEEDS = [42, 1729, 2026]
#
# Original class order:
#   [Low, Medium, High]
#
# Saved ensemble class order:
#   [High, Low, Medium]
# =========================================================

import os
import gc
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from itertools import combinations
from xgboost import XGBClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import TargetEncoder
from sklearn.metrics import balanced_accuracy_score, recall_score
from sklearn.utils.class_weight import compute_sample_weight

pd.set_option("display.max_columns", None)

# =========================================================
# 0. CONFIG
# =========================================================
TARGET = "Irrigation_Need"

TRAIN_PATH = "/kaggle/input/competitions/playground-series-s6e4/train.csv"
TEST_PATH  = "/kaggle/input/competitions/playground-series-s6e4/test.csv"
ORIG_PATH  = "/kaggle/input/datasets/miadul/irrigation-water-requirement-prediction-dataset/irrigation_prediction.csv"

SEEDS = [42, 1729, 2026]
N_SPLITS = 5

ORIG_ROW_WEIGHT = 0.5

# original notebook class order
target_map = {
    "Low": 0,
    "Medium": 1,
    "High": 2,
}
reverse_map_original = {
    0: "Low",
    1: "Medium",
    2: "High",
}

# your standard ensemble order
# original [Low, Medium, High] -> standard [High, Low, Medium]
ORIG_TO_STD = [2, 0, 1]

idx2target_std = {
    0: "High",
    1: "Low",
    2: "Medium",
}

cat_cols = [
    "Soil_Type",
    "Crop_Type",
    "Crop_Growth_Stage",
    "Season",
    "Irrigation_Type",
    "Water_Source",
    "Mulching_Used",
    "Region",
]

num_cols = [
    "Soil_pH",
    "Soil_Moisture",
    "Organic_Carbon",
    "Electrical_Conductivity",
    "Temperature_C",
    "Humidity",
    "Rainfall_mm",
    "Sunlight_Hours",
    "Wind_Speed_kmh",
    "Field_Area_hectare",
    "Previous_Irrigation_mm",
]

top_cat_cols = [
    "Crop_Growth_Stage",
    "Mulching_Used",
    "Crop_Type",
]

top_num_cols = [
    "Soil_Moisture",
    "Temperature_C",
    "Wind_Speed_kmh",
    "Rainfall_mm",
]

top_cols = [
    "Soil_Moisture",
    "Crop_Growth_Stage",
    "Temperature_C",
    "Mulching_Used",
    "Wind_Speed_kmh",
    "Rainfall_mm",
]

params_xgb_base = {
    "objective": "multi:softprob",
    "num_class": 3,
    "n_estimators": 2600,
    "learning_rate": 0.05,
    "max_depth": 3,
    "subsample": 0.9,
    "colsample_bytree": 0.8,
    "max_bin": 1100,
    "eval_metric": "mlogloss",
    "n_jobs": -1,
    "device": "cuda",
    "enable_categorical": True,
}

# =========================================================
# 1. LOAD
# =========================================================
train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)
orig = pd.read_csv(ORIG_PATH)

print("train:", train.shape)
print("test :", test.shape)
print("orig :", orig.shape)

test_ids = test["id"].copy()

for df in [train, orig]:
    df[TARGET] = df[TARGET].map(target_map).astype(np.int8)

# =========================================================
# 2. FEATURE ENGINEERING
# =========================================================
train_fe = train.copy()
test_fe = test.copy()
orig_fe = orig.copy()

def add_threshold_distances(df):
    df["soil_lt_25"] = (df["Soil_Moisture"] < 25).astype(int)
    df["wind_gt_10"] = (df["Wind_Speed_kmh"] > 10).astype(int)
    df["temp_gt_30"] = (df["Temperature_C"] > 30).astype(int)
    df["rain_lt_300"] = (df["Rainfall_mm"] < 300).astype(int)

    df["moist_rain"] = df["Soil_Moisture"] / (df["Rainfall_mm"] + 1)
    df["moist_temp"] = df["Soil_Moisture"] / (df["Temperature_C"] + 1)
    df["moist_wind"] = df["Soil_Moisture"] / (df["Wind_Speed_kmh"] + 1)
    df["ET_proxy"] = (
        df["Temperature_C"] * df["Wind_Speed_kmh"] * df["Sunlight_Hours"]
    ) / (df["Humidity"] + 1)
    df["heat_stress"] = df["Temperature_C"] * df["Sunlight_Hours"]
    df["dfrying_force"] = (
        df["Wind_Speed_kmh"] * df["Temperature_C"]
    ) / (df["Humidity"] + 1)
    df["water_supply"] = df["Rainfall_mm"] + df["Previous_Irrigation_mm"]
    df["water_dfeficit"] = df["Soil_Moisture"] - df["water_supply"] * 0.1
    df["soil_quality"] = df["Organic_Carbon"] / (
        df["Electrical_Conductivity"] + 0.1
    )
    df["moist_x_temp"] = df["Soil_Moisture"] * df["Temperature_C"]
    df["windf_x_temp"] = df["Wind_Speed_kmh"] * df["Temperature_C"]

    return df

def add_formula_features(df):
    df["high_score"] = (
        (df["Soil_Moisture"] < 25) * 2
        + (df["Rainfall_mm"] < 300) * 2
        + (df["Temperature_C"] > 30) * 1
        + (df["Wind_Speed_kmh"] > 10) * 1
    )

    df["low_score"] = (
        (df["Crop_Growth_Stage"].isin(["Harvest", "Sowing"])) * 2
        + (df["Mulching_Used"] == "Yes") * 1
    )

    df["formula_score"] = df["high_score"] - df["low_score"]

    df["formula_pred"] = 0
    df.loc[
        (df["formula_score"] > 0) & (df["formula_score"] <= 3),
        "formula_pred",
    ] = 1
    df.loc[df["formula_score"] > 3, "formula_pred"] = 2

    return df

for df in [train_fe, test_fe, orig_fe]:
    add_threshold_distances(df)
    add_formula_features(df)

# -------------------------
# N-gram categorical features
# -------------------------
BIGRAM_COLS = []
TRIGRAM_COLS = []
dataframes = [train_fe, test_fe, orig_fe]

print("\nCreating N-Gram Features...")

for c1, c2 in combinations(top_cat_cols, 2):
    col_name = f"BG_{c1}_{c2}"
    for df in dataframes:
        df[col_name] = df[c1].astype(str) + "_" + df[c2].astype(str)
    BIGRAM_COLS.append(col_name)

for c1, c2, c3 in combinations(top_cat_cols, 3):
    col_name = f"TG_{c1}_{c2}_{c3}"
    for df in dataframes:
        df[col_name] = (
            df[c1].astype(str)
            + "_"
            + df[c2].astype(str)
            + "_"
            + df[c3].astype(str)
        )
    TRIGRAM_COLS.append(col_name)

NGRAM_COLS = BIGRAM_COLS + TRIGRAM_COLS

for df in dataframes:
    df[NGRAM_COLS] = df[NGRAM_COLS].astype("category")

print(f"Created {len(NGRAM_COLS)} total N-gram features.")

for col in NGRAM_COLS:
    combined = pd.concat(
        [train_fe[col], test_fe[col], orig_fe[col]],
        axis=0,
    ).astype(str)

    labels, uniques = pd.factorize(combined)

    n_train = len(train_fe)
    n_test = len(test_fe)

    train_fe[col] = labels[:n_train]
    test_fe[col] = labels[n_train : n_train + n_test]
    orig_fe[col] = labels[n_train + n_test :]

    del combined, labels, uniques
    gc.collect()

# -------------------------
# Binned numerical x categorical interactions
# -------------------------
BIN_CAT_INT_COLS = []

for num_col in top_num_cols:
    bin_col = f"{num_col}_bin"

    train_fe[bin_col], bins = pd.qcut(
        train_fe[num_col],
        q=5,
        labels=False,
        retbins=True,
        duplicates="drop",
    )

    for df in [test_fe, orig_fe]:
        df[bin_col] = (
            pd.cut(
                df[num_col],
                bins=bins,
                labels=False,
                include_lowest=True,
            )
            .fillna(0)
            .astype(int)
        )

    for cat_col in top_cat_cols:
        int_name = f"{num_col}_bin_{cat_col}_int"

        for df in [train_fe, test_fe, orig_fe]:
            df[int_name] = df[bin_col].astype(str) + "_" + df[cat_col].astype(str)

        BIN_CAT_INT_COLS.append(int_name)

for df in [train_fe, test_fe, orig_fe]:
    df.drop(columns=[f"{num_col}_bin" for num_col in top_num_cols], inplace=True)

for col in BIN_CAT_INT_COLS:
    codes, uniques = pd.factorize(train_fe[col])
    train_fe[col] = codes.astype("int32")

    mapping = {val: i for i, val in enumerate(uniques)}

    test_fe[col] = test_fe[col].map(mapping).fillna(-1).astype("int32")
    orig_fe[col] = orig_fe[col].map(mapping).fillna(-1).astype("int32")

print(f"Created {len(BIN_CAT_INT_COLS)} binned interaction features.")

# -------------------------
# Numeric-category aggregation features
# -------------------------
NUM_CAT_AGG_COLS = []

for cat_col in top_cat_cols:
    for num_col in top_num_cols:
        group_means = train_fe.groupby(cat_col)[num_col].mean()

        avg_name = f"{num_col}_avg_by_{cat_col}"
        diff_name = f"{num_col}_diff_{cat_col}"
        ratio_name = f"{num_col}_ratio_{cat_col}"

        for df in [train_fe, test_fe, orig_fe]:
            df[avg_name] = df[cat_col].map(group_means).astype("float32")
            df[diff_name] = (df[num_col] - df[avg_name]).astype("float32")
            df[ratio_name] = (df[num_col] / (df[avg_name] + 1e-6)).astype("float32")

        NUM_CAT_AGG_COLS.extend([avg_name, diff_name, ratio_name])

NUM_CAT_AGG_COLS = list(set(NUM_CAT_AGG_COLS))
print(f"Created {len(NUM_CAT_AGG_COLS)} numeric-category agg features.")

# -------------------------
# Rounds, digits, decimals, bins
# -------------------------
round_config = {
    "Soil_Moisture": [0, -1],
    "Temperature_C": [-1],
    "Rainfall_mm": [0, -1, -2, -3],
    "Wind_Speed_kmh": [0, -1],
}

digit_config = {
    "Soil_Moisture": [-1, 0, 1, 2],
    "Temperature_C": [-1, 0, 1, 2],
    "Rainfall_mm": [-3, -2, -1, 0, 1, 2],
    "Wind_Speed_kmh": [-1, 0, 1, 2],
}

ROUND = []
for col, r_values in round_config.items():
    for r in r_values:
        feat = f"{col}_r{r}"
        for df in [train_fe, test_fe, orig_fe]:
            df[feat] = df[col].round(r)
        ROUND.append(feat)

DIGITS = []
for col, k_values in digit_config.items():
    for k in k_values:
        feat = f"{col}_d{k}"
        for df in [train_fe, test_fe, orig_fe]:
            df[feat] = ((df[col] * 10**k) % 10).astype("int16")
        DIGITS.append(feat)

DECIMALS = []
for col in top_num_cols:
    feat = f"{col}_decimal"
    for df in [train_fe, test_fe, orig_fe]:
        df[feat] = (df[col] % 1).round(2)
    DECIMALS.append(feat)

BINS = []
for col in top_num_cols:
    feat = f"{col}_bin"

    train_fe[feat], bin_edges = pd.qcut(
        train_fe[col],
        q=10,
        labels=False,
        duplicates="drop",
        retbins=True,
    )

    for df in [test_fe, orig_fe]:
        df[col] = df[col].clip(bin_edges[0], bin_edges[-1])
        df[feat] = pd.cut(
            df[col],
            bins=bin_edges,
            labels=False,
            include_lowest=True,
        ).astype(int)

    BINS.append(feat)

print(f"Created ROUND={len(ROUND)}, DIGITS={len(DIGITS)}, DECIMALS={len(DECIMALS)}, BINS={len(BINS)}")

# -------------------------
# Pairwise interactions
# -------------------------
PAIRS = []
pair_cols = top_cols

train_len = train_fe.shape[0]
test_len = test_fe.shape[0]
orig_len = orig_fe.shape[0]
combined_len = train_len + test_len + orig_len

for col1, col2 in combinations(pair_cols, 2):
    name = f"{col1}__{col2}"

    combined_str = pd.concat(
        [
            train_fe[col1].astype(str) + "_" + train_fe[col2].astype(str),
            test_fe[col1].astype(str) + "_" + test_fe[col2].astype(str),
            orig_fe[col1].astype(str) + "_" + orig_fe[col2].astype(str),
        ],
        ignore_index=True,
    )

    combined_codes, _ = pd.factorize(combined_str)
    del combined_str
    gc.collect()

    unique_count = len(np.unique(combined_codes))

    if unique_count > (combined_len // 2) or unique_count <= 1:
        del combined_codes
        gc.collect()
        continue

    train_fe[name] = combined_codes[:train_len].astype("int32")
    test_fe[name] = combined_codes[train_len : train_len + test_len].astype("int32")
    orig_fe[name] = combined_codes[train_len + test_len :].astype("int32")

    PAIRS.append(name)

    del combined_codes
    gc.collect()

print(f"{len(PAIRS)} pairwise combinations created.")

# -------------------------
# categorical dtype
# -------------------------
for df in [train_fe, test_fe, orig_fe]:
    for col in cat_cols:
        df[col] = df[col].astype("category")

# =========================================================
# 3. MAKE X / y
# =========================================================
X_train = train_fe.drop([TARGET, "id"], axis=1)
y_train = train_fe[TARGET].astype(np.int8)

X_orig = orig_fe.drop(TARGET, axis=1)
y_orig = orig_fe[TARGET].astype(np.int8)

X_test = test_fe.drop("id", axis=1)

print("\nX shapes:")
print("X_train:", X_train.shape)
print("y_train:", y_train.shape)
print("X_orig :", X_orig.shape)
print("y_orig :", y_orig.shape)
print("X_test :", X_test.shape)

DROP_COLS = PAIRS + NUM_CAT_AGG_COLS + BIN_CAT_INT_COLS
te_features = X_train.columns.tolist()

print("n DROP_COLS:", len(DROP_COLS))
print("n te_features:", len(te_features))

# =========================================================
# 4. LOG-ODDS UTILS
# =========================================================
def normalize_probs(p):
    p = np.asarray(p, dtype=np.float64)
    p = np.clip(p, 1e-15, None)
    p = p / p.sum(axis=1, keepdims=True)
    return p

def public_preds(proba, bias):
    return np.argmax(np.log(np.clip(proba, 1e-15, 1.0)) + bias, axis=1)

def apply_log_bias_proba(proba, bias, beta=1.0):
    z = np.log(np.clip(proba, 1e-15, 1.0)) + beta * bias
    z = z - z.max(axis=1, keepdims=True)
    p = np.exp(z)
    p = p / p.sum(axis=1, keepdims=True)
    return p

def tune_bias(proba, y_true):
    best_bias = np.zeros(proba.shape[1], dtype=np.float64)
    best_score = balanced_accuracy_score(y_true, public_preds(proba, best_bias))
    history = [best_score]

    for step in (1.0, 0.5, 0.2, 0.1, 0.05, 0.02, 0.01, 0.005, 0.002):
        improved = True

        while improved:
            improved = False

            for ci in range(proba.shape[1]):
                for d in (-1.0, 1.0):
                    cand = best_bias.copy()
                    cand[ci] += d * step

                    s = balanced_accuracy_score(y_true, public_preds(proba, cand))

                    if s > best_score + 1e-9:
                        best_bias = cand
                        best_score = s
                        improved = True
                        history.append(best_score)

    return best_bias, best_score, history

def class_recalls_original(y_true, pred):
    return recall_score(
        y_true,
        pred,
        labels=[0, 1, 2],
        average=None,
        zero_division=0,
    )

# =========================================================
# 5. TRAIN ONE SEED
# =========================================================
def train_one_seed(seed):
    print("\n" + "#" * 90)
    print(f"TRAIN SEED = {seed}")
    print("#" * 90)

    params_xgb = params_xgb_base.copy()
    params_xgb["random_state"] = seed

    skf = StratifiedKFold(
        n_splits=N_SPLITS,
        shuffle=True,
        random_state=seed,
    )

    oof_preds = np.zeros((len(X_train), 3), dtype=np.float64)
    test_preds = np.zeros((len(X_test), 3), dtype=np.float64)
    fold_scores = []

    for fold, (train_idx, val_idx) in enumerate(skf.split(X_train, y_train), 1):
        print(f"\n{'=' * 70}")
        print(f"Seed {seed} | Fold {fold}/{N_SPLITS}")
        print(f"{'=' * 70}")

        X_tr_real = X_train.iloc[train_idx].copy()
        X_val = X_train.iloc[val_idx].copy()

        y_tr_real = y_train.iloc[train_idx].values
        y_val = y_train.iloc[val_idx].values

        X_tr_combined = pd.concat(
            [X_tr_real, X_orig],
            axis=0,
        ).reset_index(drop=True)

        y_tr_combined = np.concatenate([y_tr_real, y_orig.values])

        encoder = TargetEncoder(
            target_type="multiclass",
            smooth="auto",
            cv=5,
            random_state=seed,
        )

        X_tr_te = encoder.fit_transform(
            X_tr_combined[te_features],
            y_tr_combined,
        )
        X_val_te = encoder.transform(X_val[te_features])
        X_test_te = encoder.transform(X_test[te_features])

        te_cols = []
        for col in te_features:
            for class_id in range(3):
                te_cols.append(f"TE_{col}_class{class_id}")

        X_tr_combined[te_cols] = X_tr_te
        X_val[te_cols] = X_val_te

        X_test_copy = X_test.copy()
        X_test_copy[te_cols] = X_test_te

        X_tr_final = X_tr_combined.drop(columns=DROP_COLS)
        X_val_final = X_val.drop(columns=DROP_COLS)
        X_te_final = X_test_copy.drop(columns=DROP_COLS)

        sample_weights = compute_sample_weight(
            "balanced",
            y_tr_combined,
        ).astype(np.float32)

        sample_weights[len(train_idx):] *= ORIG_ROW_WEIGHT

        model_xgb = XGBClassifier(**params_xgb)

        model_xgb.fit(
            X_tr_final,
            y_tr_combined,
            eval_set=[(X_val_final, y_val)],
            sample_weight=sample_weights,
            verbose=200,
        )

        val_preds_proba = model_xgb.predict_proba(X_val_final)
        test_preds_proba = model_xgb.predict_proba(X_te_final)

        oof_preds[val_idx] = val_preds_proba
        test_preds += test_preds_proba / N_SPLITS

        val_preds = np.argmax(val_preds_proba, axis=1)
        fold_ba = balanced_accuracy_score(y_val, val_preds)
        fold_scores.append(fold_ba)

        print(f"Seed {seed} Fold {fold} BA: {fold_ba:.6f}")

        del (
            model_xgb,
            X_tr_real,
            X_val,
            X_tr_combined,
            X_test_copy,
            X_tr_final,
            X_val_final,
            X_te_final,
            X_tr_te,
            X_val_te,
            X_test_te,
            sample_weights,
            val_preds_proba,
            test_preds_proba,
        )
        gc.collect()

    oof_preds = normalize_probs(oof_preds)
    test_preds = normalize_probs(test_preds)

    raw_pred = oof_preds.argmax(axis=1)
    raw_ba = balanced_accuracy_score(y_train, raw_pred)
    raw_rec = class_recalls_original(y_train, raw_pred)

    best_bias, tuned_ba, history = tune_bias(oof_preds, y_train.values)

    tuned_oof = apply_log_bias_proba(oof_preds, best_bias, beta=1.0)
    tuned_test = apply_log_bias_proba(test_preds, best_bias, beta=1.0)

    tuned_pred = tuned_oof.argmax(axis=1)
    tuned_rec = class_recalls_original(y_train, tuned_pred)

    print("\n" + "-" * 70)
    print(f"SEED {seed} RESULT")
    print("-" * 70)
    print(f"Raw OOF BA   : {raw_ba:.6f}")
    print(f"Tuned OOF BA : {tuned_ba:.6f}")
    print(f"Delta        : {tuned_ba - raw_ba:.6f}")
    print("Best bias [Low, Medium, High]:", best_bias)
    print("Raw recalls   [Low, Medium, High]:", raw_rec)
    print("Tuned recalls [Low, Medium, High]:", tuned_rec)

    # Save per-seed probabilities in original and standard order
    seed_tag = str(seed)

    np.save(f"oof_xgb_orig_domain_seed{seed_tag}_raw_original_order.npy", oof_preds.astype(np.float32))
    np.save(f"test_xgb_orig_domain_seed{seed_tag}_raw_original_order.npy", test_preds.astype(np.float32))

    np.save(f"oof_xgb_orig_domain_seed{seed_tag}_logodds_original_order.npy", tuned_oof.astype(np.float32))
    np.save(f"test_xgb_orig_domain_seed{seed_tag}_logodds_original_order.npy", tuned_test.astype(np.float32))

    np.save(f"oof_xgb_orig_domain_seed{seed_tag}_raw_std.npy", oof_preds[:, ORIG_TO_STD].astype(np.float32))
    np.save(f"test_xgb_orig_domain_seed{seed_tag}_raw_std.npy", test_preds[:, ORIG_TO_STD].astype(np.float32))

    np.save(f"oof_xgb_orig_domain_seed{seed_tag}_logodds_std.npy", tuned_oof[:, ORIG_TO_STD].astype(np.float32))
    np.save(f"test_xgb_orig_domain_seed{seed_tag}_logodds_std.npy", tuned_test[:, ORIG_TO_STD].astype(np.float32))

    return {
        "seed": seed,
        "oof_raw": oof_preds,
        "test_raw": test_preds,
        "oof_tuned": tuned_oof,
        "test_tuned": tuned_test,
        "raw_ba": raw_ba,
        "tuned_ba": tuned_ba,
        "best_bias": best_bias,
        "fold_scores": fold_scores,
    }

# =========================================================
# 6. RUN MULTI SEED
# =========================================================
all_results = []

for seed in SEEDS:
    res = train_one_seed(seed)
    all_results.append(res)
    gc.collect()

# =========================================================
# 7. MULTI-SEED ENSEMBLE
# =========================================================
print("\n" + "#" * 90)
print("MULTI-SEED ENSEMBLE")
print("#" * 90)

oof_raw_avg = normalize_probs(
    np.mean([r["oof_raw"] for r in all_results], axis=0)
)
test_raw_avg = normalize_probs(
    np.mean([r["test_raw"] for r in all_results], axis=0)
)

oof_tuned_avg = normalize_probs(
    np.mean([r["oof_tuned"] for r in all_results], axis=0)
)
test_tuned_avg = normalize_probs(
    np.mean([r["test_tuned"] for r in all_results], axis=0)
)

raw_avg_pred = oof_raw_avg.argmax(axis=1)
raw_avg_ba = balanced_accuracy_score(y_train, raw_avg_pred)
raw_avg_rec = class_recalls_original(y_train, raw_avg_pred)

tuned_avg_pred = oof_tuned_avg.argmax(axis=1)
tuned_avg_ba = balanced_accuracy_score(y_train, tuned_avg_pred)
tuned_avg_rec = class_recalls_original(y_train, tuned_avg_pred)

# recommended: tune bias on averaged raw probabilities
ens_bias, ens_tuned_ba, ens_history = tune_bias(oof_raw_avg, y_train.values)
oof_rawavg_logodds = apply_log_bias_proba(oof_raw_avg, ens_bias, beta=1.0)
test_rawavg_logodds = apply_log_bias_proba(test_raw_avg, ens_bias, beta=1.0)

rawavg_logodds_pred = oof_rawavg_logodds.argmax(axis=1)
rawavg_logodds_rec = class_recalls_original(y_train, rawavg_logodds_pred)

print("\nSingle seed summary:")
seed_rows = []
for r in all_results:
    seed_rows.append({
        "seed": r["seed"],
        "raw_ba": r["raw_ba"],
        "tuned_ba": r["tuned_ba"],
        "delta": r["tuned_ba"] - r["raw_ba"],
        "best_bias": r["best_bias"],
    })
seed_df = pd.DataFrame(seed_rows)
display(seed_df)

print("\nAveraged raw probs:")
print(f"OOF BA: {raw_avg_ba:.6f}")
print("recalls [Low, Medium, High]:", raw_avg_rec)

print("\nAveraged per-seed tuned probs:")
print(f"OOF BA: {tuned_avg_ba:.6f}")
print("recalls [Low, Medium, High]:", tuned_avg_rec)

print("\nRaw-average + ensemble logodds:")
print("ensemble bias [Low, Medium, High]:", ens_bias)
print(f"OOF BA: {ens_tuned_ba:.6f}")
print("recalls [Low, Medium, High]:", rawavg_logodds_rec)

# =========================================================
# 8. SAVE FINAL PROBS
# =========================================================
# Save in original order
np.save("oof_xgb_orig_domain_multiseed_rawavg_original_order.npy", oof_raw_avg.astype(np.float32))
np.save("test_xgb_orig_domain_multiseed_rawavg_original_order.npy", test_raw_avg.astype(np.float32))

np.save("oof_xgb_orig_domain_multiseed_tunedavg_original_order.npy", oof_tuned_avg.astype(np.float32))
np.save("test_xgb_orig_domain_multiseed_tunedavg_original_order.npy", test_tuned_avg.astype(np.float32))

np.save("oof_xgb_orig_domain_multiseed_rawavg_logodds_original_order.npy", oof_rawavg_logodds.astype(np.float32))
np.save("test_xgb_orig_domain_multiseed_rawavg_logodds_original_order.npy", test_rawavg_logodds.astype(np.float32))

# Save in your standard order [High, Low, Medium]
np.save("oof_xgb_orig_domain_multiseed_rawavg_std.npy", oof_raw_avg[:, ORIG_TO_STD].astype(np.float32))
np.save("test_xgb_orig_domain_multiseed_rawavg_std.npy", test_raw_avg[:, ORIG_TO_STD].astype(np.float32))

np.save("oof_xgb_orig_domain_multiseed_tunedavg_std.npy", oof_tuned_avg[:, ORIG_TO_STD].astype(np.float32))
np.save("test_xgb_orig_domain_multiseed_tunedavg_std.npy", test_tuned_avg[:, ORIG_TO_STD].astype(np.float32))

np.save("oof_xgb_orig_domain_multiseed_rawavg_logodds_std.npy", oof_rawavg_logodds[:, ORIG_TO_STD].astype(np.float32))
np.save("test_xgb_orig_domain_multiseed_rawavg_logodds_std.npy", test_rawavg_logodds[:, ORIG_TO_STD].astype(np.float32))

seed_df.to_csv("xgb_orig_domain_multiseed_seed_summary.csv", index=False)

summary_df = pd.DataFrame({
    "model": [
        "multiseed_rawavg",
        "multiseed_tunedavg",
        "multiseed_rawavg_logodds",
    ],
    "oof_ba": [
        raw_avg_ba,
        tuned_avg_ba,
        ens_tuned_ba,
    ],
})
summary_df.to_csv("xgb_orig_domain_multiseed_summary.csv", index=False)

# =========================================================
# 9. SUBMISSIONS
# =========================================================
def make_submission_from_original_order_probs(probs, filename):
    pred = probs.argmax(axis=1)
    sub = pd.DataFrame({
        "id": test_ids,
        TARGET: pd.Series(pred).map(reverse_map_original),
    })
    sub.to_csv(filename, index=False)

    print("\n", filename)
    print(sub[TARGET].value_counts(normalize=True).sort_index())
    print(sub.head())

    return sub

sub_rawavg = make_submission_from_original_order_probs(
    test_raw_avg,
    "submission_xgb_orig_domain_multiseed_rawavg.csv",
)

sub_tunedavg = make_submission_from_original_order_probs(
    test_tuned_avg,
    "submission_xgb_orig_domain_multiseed_tunedavg.csv",
)

sub_rawavg_logodds = make_submission_from_original_order_probs(
    test_rawavg_logodds,
    "submission_xgb_orig_domain_multiseed_rawavg_logodds.csv",
)

# default submission.csv は一番おすすめの rawavg_logodds
sub_rawavg_logodds.to_csv("submission.csv", index=False)

print("\nSaved final files:")
print("- submission.csv")
print("- submission_xgb_orig_domain_multiseed_rawavg.csv")
print("- submission_xgb_orig_domain_multiseed_tunedavg.csv")
print("- submission_xgb_orig_domain_multiseed_rawavg_logodds.csv")
print("- oof/test_xgb_orig_domain_multiseed_*_std.npy")
print("- xgb_orig_domain_multiseed_seed_summary.csv")
print("- xgb_orig_domain_multiseed_summary.csv")

train: (630000, 21)
test : (270000, 20)
orig : (10000, 20)

Creating N-Gram Features...
Created 4 total N-gram features.
Created 12 binned interaction features.
Created 36 numeric-category agg features.
Created ROUND=9, DIGITS=18, DECIMALS=4, BINS=4
9 pairwise combinations created.

X shapes:
X_train: (630000, 134)
y_train: (630000,)
X_orig : (10000, 134)
y_orig : (10000,)
X_test : (270000, 134)
n DROP_COLS: 57
n te_features: 134

##########################################################################################
TRAIN SEED = 42
##########################################################################################

Seed 42 | Fold 1/5
[0]	validation_0-mlogloss:1.02976
[200]	validation_0-mlogloss:0.06296
[400]	validation_0-mlogloss:0.05696
[600]	validation_0-mlogloss:0.05462
[800]	validation_0-mlogloss:0.05310
[1000]	validation_0-mlogloss:0.05201
[1200]	validation_0-mlogloss:0.05123
[1400]	validation_0-mlogloss:0.05062
[1600]	validation_0-mlogloss:0.05006
[1800]	validation_0-m

,seed,raw_ba,tuned_ba,delta,best_bias
0,42,0.978269,0.980106,0.001837,"[-1.2999999999999998, -1.195, 0.0]"
1,1729,0.978131,0.979766,0.001635,"[-1.288, -1.295, 0.0]"
2,2026,0.978227,0.979762,0.001535,"[-1.0919999999999999, -1.13, 0.0]"



Averaged raw probs:
OOF BA: 0.978649
recalls [Low, Medium, High]: [0.99498266 0.97542602 0.96553858]

Averaged per-seed tuned probs:
OOF BA: 0.979955
recalls [Low, Medium, High]: [0.99491778 0.96826924 0.97667666]

Raw-average + ensemble logodds:
ensemble bias [Low, Medium, High]: [-1.103 -0.905  0.   ]
OOF BA: 0.980229
recalls [Low, Medium, High]: [0.99446903 0.97068272 0.97553429]

 submission_xgb_orig_domain_multiseed_rawavg.csv
Irrigation_Need
High      0.034737
Low       0.590707
Medium    0.374556
Name: proportion, dtype: float64
       id Irrigation_Need
0  630000             Low
1  630001             Low
2  630002             Low
3  630003             Low
4  630004             Low

 submission_xgb_orig_domain_multiseed_tunedavg.csv
Irrigation_Need
High      0.038122
Low       0.590667
Medium    0.371211
Name: proportion, dtype: float64
       id Irrigation_Need
0  630000             Low
1  630001             Low
2  630002             Low
3  630003             Low
4  630004    